# 02 — M2 training (Kaggle GPU T4 x2)
Accelerator: **GPU T4 x2**. Internet: **On**. Inputs: notebook output **traffic-m1-data-prep**. Secret: **HF_TOKEN** attached.

Interactive session: run cells 1–4 (timing run). For the full run set `RUN_TIMING = False` and `EPOCHS`, then **Save & Run All (Commit)**.

In [ ]:
# 0. Settings
RUN_TIMING = True      # True: 1-epoch timing run only. False: full training + push to HF.
EPOCHS = 40            # set from the measured epoch time before the full run
DEVICE = '0,1'         # fall back to '0' if DDP fails
HF_REPO = 'andromedaboy/traffic-analytics-yolo11n'

In [ ]:
# 1. Clone + install
%cd /kaggle/working
!rm -rf traffic-analytics
!git clone --depth 1 https://github.com/boyatomic83/traffic-analytics.git
%cd /kaggle/working/traffic-analytics
!git log -1 --oneline
!pip install -q -e ".[train]"

In [ ]:
# 2. Environment check
!nvidia-smi --query-gpu=name,memory.total --format=csv
!python -c "import torch, ultralytics, traffic_analytics.train_utils as t; print('torch', torch.__version__, 'cuda', torch.cuda.is_available(), 'gpus', torch.cuda.device_count()); print('ultralytics', ultralytics.__version__); print(t.__file__)"
!python -c "from traffic_analytics.train_utils import find_dataset_root; print('DATA =', find_dataset_root('/kaggle/input'))"

In [ ]:
# 3. Timing run: 1 epoch on the full data (only when RUN_TIMING)
if RUN_TIMING:
    !python scripts/train.py --search /kaggle/input --out /tmp/m2_timing --epochs 1 --device {DEVICE} > /kaggle/working/timing.log 2>&1; echo "EXIT CODE: $?"
    !grep -E 'DDP|Epoch|all |car |van |bus |truck_other|INFO|ERROR|Error' /kaggle/working/timing.log | tail -n 40
    !tail -n 25 /kaggle/working/timing.log
    !cat /tmp/m2_timing/train/results.csv

In [ ]:
# 4. Full training (only when not RUN_TIMING)
if not RUN_TIMING:
    !python scripts/train.py --search /kaggle/input --out /kaggle/working/m2 --epochs {EPOCHS} --device {DEVICE} > /kaggle/working/train.log 2>&1; echo "EXIT CODE: $?"
    !grep -E 'INFO|ERROR|Error' /kaggle/working/train.log | tail -n 30
    !tail -n 25 /kaggle/working/train.log
    !ls -la /kaggle/working/m2/artifacts

In [ ]:
# 5. Push artifacts to the Hugging Face model repo (only when not RUN_TIMING)
import os
if not RUN_TIMING:
    from kaggle_secrets import UserSecretsClient
    os.environ['HF_TOKEN'] = UserSecretsClient().get_secret('HF_TOKEN')
    !python scripts/push_to_hf.py --repo-id {HF_REPO} --folder /kaggle/working/m2/artifacts --path-in-repo m2 --message "M2: fine-tuned YOLO11n (best.pt, charts, metrics)" 2>&1 | tail -n 20